# Ablation Study — CIFAR-10 CNN

5 experiments, each comparing variations of a single factor.
Run each cell independently — they share only the setup cell.

In [ ]:
# Setup — run this first
!git clone https://github.com/declspecl/csi4140.git 2>/dev/null; cd csi4140 && git pull
import sys, os

os.chdir("/content/csi4140")
!pip install -e . -q

import torch, json
import matplotlib.pyplot as plt
from src.data.cifar10 import get_cifar10_dataloaders
from src.models.cifar10_cnn import build_cifar10_cnn
from src.network.layer.convolutional import Convolutional
from src.network.layer.activation_layer import ActivationLayer
from src.network.layer.flatten import Flatten
from src.network.layer.fully_connected import FullyConnected
from src.network.layer.dropout import Dropout
from src.network.activation.relu import ReLU
from src.network.activation.softmax import Softmax
from src.network.neural_network import NeuralNetwork
from src.network.loss.cross_entropy import CrossEntropy
from src.network.optimizer.adam import Adam
from src.network.optimizer.momentum import Momentum
from src.network.scheduler.cosine import CosineDecay
from src.network.scheduler.step import StepDecay
from src.network.regularizer.l2 import L2
from src.train import train

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 10
RESULTS_DIR = "results/ablation"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load data once, reuse everywhere
train_loader, test_loader = get_cifar10_dataloaders(batch_size=128, num_workers=2)
loss_fn = CrossEntropy()


def run(name, model, optimizer, scheduler=None):
    model.to(DEVICE)
    h = train(
        model,
        train_loader,
        test_loader,
        loss_fn,
        optimizer,
        scheduler,
        epochs=EPOCHS,
        device=DEVICE,
        checkpoint_path=os.path.join(RESULTS_DIR, f"{name}.pt"),
    )
    return h


def plot_exp(results, title, save_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    for label, h in results.items():
        epochs = range(1, len(h["test_acc"]) + 1)
        ax1.plot(epochs, h["test_acc"], label=label)
        ax2.plot(epochs, h["train_loss"], label=label)
    ax1.set(xlabel="Epoch", ylabel="Test Accuracy", title=f"{title} — Test Accuracy")
    ax1.legend()
    ax1.grid(True)
    ax2.set(xlabel="Epoch", ylabel="Training Loss", title=f"{title} — Training Loss")
    ax2.legend()
    ax2.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"{save_name}.png"), dpi=300, bbox_inches="tight")
    plt.show()
    # Save JSON
    with open(os.path.join(RESULTS_DIR, f"{save_name}.json"), "w") as f:
        json.dump(
            {k: {kk: vv for kk, vv in v.items() if kk != "iteration_loss"} for k, v in results.items()}, f, indent=2
        )


def build_dropout_model(p=0.3):
    return NeuralNetwork(
        [
            Convolutional(3, 32, kernel_size=3, stride=1, padding=1),
            ActivationLayer(ReLU()),
            Dropout(p=p),
            Convolutional(32, 64, kernel_size=3, stride=2, padding=1),
            ActivationLayer(ReLU()),
            Dropout(p=p),
            Convolutional(64, 128, kernel_size=3, stride=2, padding=1),
            ActivationLayer(ReLU()),
            Dropout(p=p),
            Flatten(),
            FullyConnected(128 * 8 * 8, 512, ReLU()),
            FullyConnected(512, 10, Softmax()),
        ]
    )


print(f"Device: {DEVICE} | Epochs: {EPOCHS} | Batches/epoch: {len(train_loader)}")

## Experiment 1: Learning Rate Decay

In [ ]:
exp1 = {}

m = build_cifar10_cnn()
o = Adam(m.parameters(), learning_rate=0.001)
exp1["Cosine Decay"] = run("exp1_cosine", m, o, CosineDecay(o, epochs=EPOCHS))

m = build_cifar10_cnn()
o = Adam(m.parameters(), learning_rate=0.001)
exp1["Step Decay (×0.5 / 5ep)"] = run("exp1_step", m, o, StepDecay(o, step_size=5, decay_factor=0.5))

m = build_cifar10_cnn()
o = Adam(m.parameters(), learning_rate=0.001)
exp1["No Decay"] = run("exp1_none", m, o, None)

plot_exp(exp1, "LR Decay", "exp1_lr_decay")

## Experiment 2: Regularization Methods

In [ ]:
exp2 = {}

m = build_cifar10_cnn()
o = Adam(m.parameters(), learning_rate=0.001)
exp2["None"] = run("exp2_none", m, o)

m = build_cifar10_cnn()
o = Adam(m.parameters(), learning_rate=0.001, regularizer=L2(lambda_=0.001))
exp2["L2 (λ=0.001)"] = run("exp2_l2", m, o)

m = build_dropout_model(p=0.3)
o = Adam(m.parameters(), learning_rate=0.001)
exp2["Dropout (p=0.3)"] = run("exp2_dropout", m, o)

m = build_dropout_model(p=0.3)
o = Adam(m.parameters(), learning_rate=0.001, regularizer=L2(lambda_=0.001))
exp2["L2 + Dropout"] = run("exp2_both", m, o)

plot_exp(exp2, "Regularization", "exp2_regularization")

## Experiment 3: L2 Lambda

In [ ]:
exp3 = {}

for lam in [0.0001, 0.001, 0.01, 0.1]:
    m = build_cifar10_cnn()
    o = Adam(m.parameters(), learning_rate=0.001, regularizer=L2(lambda_=lam))
    exp3[f"λ={lam}"] = run(f"exp3_l2_{lam}", m, o)

plot_exp(exp3, "L2 Lambda", "exp3_l2_lambda")

## Experiment 4: Optimizer Comparison

In [ ]:
exp4 = {}

m = build_cifar10_cnn()
o = Adam(m.parameters(), learning_rate=0.001)
exp4["Adam (lr=0.001)"] = run("exp4_adam", m, o)

m = build_cifar10_cnn()
o = Momentum(m.parameters(), learning_rate=0.01, beta=0.9)
exp4["SGD+Momentum (lr=0.01)"] = run("exp4_mom_01", m, o)

m = build_cifar10_cnn()
o = Momentum(m.parameters(), learning_rate=0.05, beta=0.9)
exp4["SGD+Momentum (lr=0.05)"] = run("exp4_mom_05", m, o)

plot_exp(exp4, "Optimizer", "exp4_optimizer")

## Experiment 5: Adam Beta Parameters

In [ ]:
exp5 = {}

for label, b1, b2 in [
    ("β1=0.9, β2=0.999 (default)", 0.9, 0.999),
    ("β1=0.95, β2=0.999", 0.95, 0.999),
    ("β1=0.8, β2=0.999", 0.8, 0.999),
    ("β1=0.9, β2=0.99", 0.9, 0.99),
    ("β1=0.9, β2=0.9", 0.9, 0.9),
]:
    m = build_cifar10_cnn()
    o = Adam(m.parameters(), learning_rate=0.001, beta1=b1, beta2=b2)
    exp5[label] = run(f"exp5_b1{b1}_b2{b2}", m, o)

plot_exp(exp5, "Adam Betas", "exp5_adam_betas")